# Advanced 03 — Governance and Production Readiness

Validate typed, owned, fresh evidence and issue a release decision that cannot average away severe failures.

## 1. Load the course lab

The notebook imports the reusable course module rather than copying its security logic.

In [ ]:
import runpy
from datetime import datetime, timedelta, timezone
ns = runpy.run_path('03_production_gate.py')
Evidence, RiskAcceptance, REQUIRED_EVIDENCE, evaluate_release = (ns[name] for name in ('Evidence','RiskAcceptance','REQUIRED_EVIDENCE','evaluate_release'))
now = datetime(2026,9,12,tzinfo=timezone.utc)
evidence = [Evidence(kind,f'artifact:{kind}','release-7','team:agent-security',now-timedelta(days=1),True) for kind in REQUIRED_EVIDENCE]
risks = [RiskAcceptance('R-12','director:platform','bounded read-only pilot latency risk',now+timedelta(days=14))]

## 2. Establish the safe baseline

Observe the trusted inputs and the decision evidence before injecting failures.

In [ ]:
decision = evaluate_release(evidence,severe_attack_successes=0,residual_risks=risks,now=now)
assert decision.ready and not decision.blockers
decision

## 3. Inject an attack

Change one security-relevant boundary and keep the rest of the fixture stable.

In [ ]:
severe = evaluate_release(evidence,severe_attack_successes=1,residual_risks=risks,now=now)
assert not severe.ready and 'severe-attack-successes:1' in severe.blockers
severe

## 4. Attempt a bypass

The assertions below make the security property executable and regression-testable.

In [ ]:
stale = list(evidence)
stale[0] = Evidence(stale[0].kind,stale[0].value,stale[0].version,stale[0].owner,now-timedelta(days=60),True)
stale_decision = evaluate_release(stale,severe_attack_successes=0,residual_risks=risks,now=now)
assert any(item.startswith('stale:') for item in stale_decision.blockers)

## 5. Evaluate observable outcomes

Use explicit denominators or counts. Private model reasoning is neither required nor recorded.

In [ ]:
{'ready': decision.ready, 'evidence_count': len(evidence), 'version_bindings': decision.evidence_versions, 'receipt_id': decision.receipt_id}

## 6. Exercise a second failure mode

In [ ]:
expired = [RiskAcceptance('R-12','director:platform','risk still exists',now-timedelta(seconds=1))]
expired_decision = evaluate_release(evidence,severe_attack_successes=0,residual_risks=expired,now=now)
assert not expired_decision.ready

## 7. Production replacement

Production replacement: authenticated evidence producers, protected history, branch and environment controls, staged deployment, monitored rollback, kill switches, accountable exceptions, and periodic reapproval after material change.

## Checkpoint

Explain which trusted component enforces the invariant, what evidence proves the decision, and what residual risk remains.